# UV-Vis Redox Titration Workup

This notebook combines the two original MATLAB scripts into one workflow, plus a curve-fitting
step:

1. **Part 1 — `.SPC` → `Full_data.xlsx`** (originally `asc_to_xlsx_Voltage.m`, now reading `.SPC`
   directly instead of a separate ASCII-conversion step): parses each `.SPC` spectrum straight to
   `.csv`, tags each with its electrode voltage, baseline-subtracts each spectrum, sorts by
   voltage, and writes a combined spreadsheet. Includes a full-range and a zoomed-in spectra plot
   with dashed guide lines at your candidate isosbestic/apex wavelengths, to help you pick them.
2. **Part 2 — Fraction vs. Potential** (from `data_analyse.m`): uses the isosbestic and apex
   wavelengths to compute the reduced/oxidized fraction at each voltage and writes
   `PotentialFraction.xlsx`.
3. **Part 3 — Curve fit**: fits the one- or two-potential Boltzmann/Nernst equation to the
   Potential/Fraction data with `lmfit` (Python's equivalent of MATLAB's Curve Fitting app), and
   saves everything needed later (equation, fitted parameters, fit data) to `oxi.xlsx`/`red.xlsx`.

A separate script, `plot_together.py`, reads the `oxi.xlsx` and `red.xlsx` files this notebook
produces (once you've copied both into the same folder) and reproduces the combined
oxidative/reductive plot from `Echem_plot_together.m`. It's intentionally kept outside this
notebook -- run it independently once both files exist.

**All editable parameters live in the single cell right below this one.** You shouldn't need to
change anything else in the notebook — just run top to bottom.

See `README.md` for the full step-by-step workflow (including how this notebook and
`plot_together.py` fit together).

In [ ]:
# =================================================================================
# ALL PARAMETERS -- edit everything here, then run the notebook top to bottom.
# =================================================================================

# ---- Part 1: .asc -> Full_data.xlsx ----------------------------------------------
DATA_DIR = "."            # folder containing the .asc files
Electrode_Voltage = 187    # electrode voltage offset added to each filename's voltage (mV)

# ---- Part 1/2: candidate isosbestic wavelengths and apex -------------------------
# These are just the STARTING positions for the interactive sliders in Part 1 below --
# drag the sliders there to fine-tune; you don't need to get these exact here.
isosbestic1 = 538.5
isosbestic2 = 559
Apex = 550.5

# ---- Part 3: curve fit ------------------------------------------------------------
num_potentials = 1   # 1 = single reduction potential, 2 = two overlapping potentials

# Initial guesses for the first (or only) wave. A1_guess defaults to 1.0 for a single
# potential (the fraction data spans 0-1) and 0.5 for two potentials (each wave covers
# roughly half the total range) -- override either if you have a better starting guess.
Ema_guess = 200       # initial guess for the first (or only) midpoint potential, mV
n1_guess = 1          # initial guess for the electron count of the first wave
A1_guess = 1.0 if num_potentials == 1 else 0.5   # initial guess for the amplitude of the first wave

# Initial guesses for the second wave -- only used (and only need to be set) when
# num_potentials == 2.
if num_potentials == 2:
    Emb_guess = 0     # initial guess for the second midpoint potential, mV
    n2_guess = 1        # initial guess for the electron count of the second wave
    A2_guess = 0.5      # initial guess for the amplitude of the second wave

assert num_potentials in (1, 2), "num_potentials must be 1 or 2"

# ---- Part 3: output naming --------------------------------------------------------
# Set this to "oxi" for an oxidative sweep or "red" for a reductive sweep -- the fitted
# results are saved to "<sweep_label>.xlsx", which plot_together.py later reads.
sweep_label = "oxi"
# =================================================================================


In [ ]:
# Standard library
import os
import glob
import struct

# Third-party
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from lmfit import Model, conf_interval


## Part 1 — Convert `.SPC` files to `Full_data.xlsx`

**Important (same warning as the original script, just for `.SPC` now):** name your `.SPC` files
using *only* their voltage (e.g. `-200.SPC`, `0.SPC`, `150.SPC`). Files with other characters in
the name won't sort or be tagged correctly.

`.SPC` is Thermo/Galactic's old binary spectrum format. There's no official spec available here,
so the parser below (`parse_spc`) was reverse-engineered directly from a sample file's bytes:
wavelength range and point count are read from fixed header offsets, and the data is checked to
exactly fill the rest of the file -- if a file doesn't match that layout, it raises a clear error
rather than silently producing a wrong wavelength axis. If that happens on one of your files,
it likely means this instrument's `.SPC` variant differs slightly and the offsets need adjusting.

This part baseline-subtracts every spectrum by its **first** data row. The original MATLAB script
forced this to correspond to 600 nm — make sure the first row of your spectra is the wavelength
you want to use as the baseline reference, or edit the "Baseline subtraction" cell below directly
if that's not the case for your data.

*(The interactive wavelength picker below requires `ipywidgets`: `pip install ipywidgets` if it
isn't already installed, then restart the kernel.)*

In [ ]:
def parse_spc(path):
    """Parse one old-format .SPC binary file into (wavelengths, absorbance) arrays.

    Reverse-engineered byte layout (see markdown above for how/why):
        offset 10 (float32): ffirst -- first wavelength
        offset 14 (float32): flast  -- last wavelength
        offset 118 (int16):  fnpts  -- number of data points
        offset 120 onward:   fnpts x float32 -- absorbance values, ending exactly at EOF

    The wavelength axis isn't stored point-by-point -- it's evenly spaced between ffirst
    and flast over fnpts points.
    """
    with open(path, "rb") as fh:
        data = fh.read()

    ffirst = struct.unpack("<f", data[10:14])[0]
    flast = struct.unpack("<f", data[14:18])[0]
    fnpts = struct.unpack("<h", data[118:120])[0]

    expected_bytes = 4 * fnpts
    actual_bytes = len(data) - 120
    if actual_bytes != expected_bytes:
        raise ValueError(
            f"{path}: header says {fnpts} points ({expected_bytes} bytes), but "
            f"{actual_bytes} bytes remain after the header -- this file's layout may "
            f"not match the one this parser was reverse-engineered from."
        )

    absorbance = np.array(struct.unpack("<" + "f" * fnpts, data[120:120 + expected_bytes]))
    wavelengths = np.linspace(ffirst, flast, fnpts)
    return wavelengths, absorbance


# Work in the folder containing the raw .SPC files for the rest of Part 1.
os.chdir(DATA_DIR)

# Parse every .SPC file here directly to .csv (no separate ASCII-conversion step needed)
# -- writes "<name>.csv" for each "<name>.SPC", matching the voltage-only naming the rest
# of Part 1 expects.
spc_files = sorted(glob.glob("*.SPC")) + sorted(glob.glob("*.spc"))
for f in spc_files:
    wavelengths, absorbance = parse_spc(f)
    csv_name = os.path.splitext(f)[0] + ".csv"
    with open(csv_name, "w") as fh:
        for w, a in zip(wavelengths, absorbance):
            fh.write(f"{w:.1f},{a:.5f}\n")

print(f"Converted {len(spc_files)} .SPC files to .csv")


In [ ]:
def read_two_col(path):
    """Read a 2-column (wavelength, absorbance) text file, comma- or
    whitespace-delimited -- mirrors MATLAB's auto-detecting readmatrix."""
    try:
        data = np.loadtxt(path, delimiter=",")
    except ValueError:
        data = np.loadtxt(path)
    return data


def _is_voltage_filename(path):
    """True if the filename (without extension) is a plain number, e.g. '-200.csv'.

    Filters out this notebook's own output files (Full_data.xlsx, PotentialFraction.xlsx,
    oxi.xlsx/red.xlsx, etc.) if they happen to sit in the same folder as the raw data --
    without this, a second run would try to parse its own previous output as a voltage.
    """
    stem = os.path.splitext(os.path.basename(path))[0]
    try:
        float(stem)
        return True
    except ValueError:
        return False


# Read every voltage-named .csv in the current directory, tag each column with its
# voltage (filename + Electrode_Voltage), and stack the absorbance columns side by side.
csv_files = sorted(f for f in glob.glob("*.csv") if _is_voltage_filename(f))
Filenum = len(csv_files)

fileName = []     # one voltage per file, in csv_files order
Absorbance = []   # one absorbance column per file, in the same order
for f in csv_files:
    voltage = float(os.path.splitext(f)[0]) + Electrode_Voltage
    fileName.append(voltage)
    Temp = read_two_col(f)
    Absorbance.append(Temp[:, 1])

fileName = np.array(fileName)
Absorbance = np.column_stack(Absorbance)   # rows = wavelength points, cols = files

UnsortedMatrix = np.vstack([fileName, Absorbance])   # row 0 = voltages, rest = spectra
print("UnsortedMatrix shape:", UnsortedMatrix.shape)


In [ ]:
# ---- Baseline subtraction ---------------------------------------------------------
# Subtract every column by its own first absorbance value (row index 1, i.e. the row
# right after the voltage row). The original MATLAB script hardcoded this to be the
# 600 nm point ("force baseline aligning all data at 600 nm") -- it assumes the first
# row of your data is at that wavelength. Comment this cell out entirely if you don't
# want baseline subtraction.
baseline = UnsortedMatrix[1, :]
UnsortedMatrix[1:, :] = UnsortedMatrix[1:, :] - baseline


In [ ]:
# Sort columns left-to-right by voltage (row 0), so the spreadsheet and plots read in
# a sensible order regardless of the order .asc files were found in.
sort_idx = np.argsort(UnsortedMatrix[0, :])
SortedMatrix = UnsortedMatrix[:, sort_idx]

# Prepend the wavelength column. -233333 is a placeholder in the very top-left cell
# (same trick as the MATLAB script) -- it has no physical meaning, it just marks "this
# is the voltage row" so Part 2 can find it later by searching column 0 for -233333,
# the same way the MATLAB script used find() to locate rows by their wavelength value.
Wavelength_col = read_two_col(csv_files[0])[:, 0]
Wavelength = np.concatenate([[-233333], Wavelength_col])

ResultMatrix = np.column_stack([Wavelength, SortedMatrix])
x = ResultMatrix[1:, 0]   # wavelength axis, reused by the plots below
print("ResultMatrix shape:", ResultMatrix.shape)


**Drag the sliders below** to place `isosbestic1`, `isosbestic2`, and `Apex`. Both plots
(full-range and zoomed) redraw when you release the slider — the zoom window follows automatically.
(Redraw is on release rather than continuous, so dragging stays smooth even with many spectra.)

In [ ]:
# One slider per wavelength of interest, seeded from the parameters cell above.
iso1_slider = widgets.FloatSlider(value=isosbestic1, min=float(x.min()), max=float(x.max()),
                                   step=0.5, description="isosbestic1", readout_format=".1f",
                                   continuous_update=False,
                                   style={"description_width": "90px"}, layout=widgets.Layout(width="500px"))
iso2_slider = widgets.FloatSlider(value=isosbestic2, min=float(x.min()), max=float(x.max()),
                                   step=0.5, description="isosbestic2", readout_format=".1f",
                                   continuous_update=False,
                                   style={"description_width": "90px"}, layout=widgets.Layout(width="500px"))
apex_slider = widgets.FloatSlider(value=Apex, min=float(x.min()), max=float(x.max()),
                                   step=0.5, description="Apex", readout_format=".1f",
                                   continuous_update=False,
                                   style={"description_width": "90px"}, layout=widgets.Layout(width="500px"))


def _plot_with_guides(iso1, iso2, apex):
    """Redraw the full-range and zoomed spectra with dashed guide lines at the three
    slider positions. Called automatically by widgets.interact whenever a slider is
    released (continuous_update=False keeps dragging itself smooth)."""
    zoom_margin = 15   # nm of padding either side of the three wavelengths, for the zoomed plot
    zoom_xmin = min(iso1, iso2, apex) - zoom_margin
    zoom_xmax = max(iso1, iso2, apex) + zoom_margin

    fig, (ax_full, ax_zoom) = plt.subplots(1, 2, figsize=(13, 5))
    windows = [(ax_full, (float(x.min()), float(x.max())), "Full range"),
               (ax_zoom, (zoom_xmin, zoom_xmax), f"Zoomed ({zoom_xmin:.0f}-{zoom_xmax:.0f} nm)")]

    for ax, (xmin, xmax), title in windows:
        for i in range(1, Filenum + 1):
            ax.plot(x, ResultMatrix[1:, i])
        ax.set_xlim(xmin, xmax)

        # Scale the guide-line labels to the tallest trace actually visible in this
        # window, so they sit just above the data rather than off in empty space.
        mask = (x >= xmin) & (x <= xmax)
        ymax = ResultMatrix[1:, 1:][mask].max() if mask.any() else ResultMatrix[1:, 1:].max()
        for wl, label in [(iso1, "isosbestic 1"), (iso2, "isosbestic 2"), (apex, "apex")]:
            ax.axvline(wl, color="black", linestyle="--", linewidth=1)
            ax.text(wl, ymax, f" {label} ({wl:.1f} nm)", rotation=90, va="top", ha="right", fontsize=8)

        ax.set_xlabel("Wavelength (nm)")
        ax.set_ylabel("Absorbance")
        ax.set_title(title)

    plt.tight_layout()
    plt.show()
    plt.close(fig)   # avoid piling up figures in memory across repeated redraws


widgets.interact(_plot_with_guides, iso1=iso1_slider, iso2=iso2_slider, apex=apex_slider)


In [ ]:
# Run this cell once you're happy with the slider positions above -- it copies the
# current slider values into isosbestic1 / isosbestic2 / Apex, for use in Part 2 and
# Part 3 below. (Deliberately a separate, manual step rather than auto-syncing, so you
# don't accidentally lock in a value mid-drag.)
isosbestic1 = iso1_slider.value
isosbestic2 = iso2_slider.value
Apex = apex_slider.value
print(f"Locked in: isosbestic1={isosbestic1}, isosbestic2={isosbestic2}, Apex={Apex}")


In [ ]:
pd.DataFrame(ResultMatrix).to_excel("Full_data.xlsx", header=False, index=False)
print("Saved Full_data.xlsx")


## Part 2 — Fraction vs. Potential

Fits/evaluates:

`Fraction(V) = (Modified_Apex(V) - Modified_Apex_end) / (Modified_Apex_1 - Modified_Apex_end)`

using the `isosbestic1`, `isosbestic2`, and `Apex` wavelengths set in the parameters cell (and
checked against the dashed guide lines on the Part 1 plots), exactly as in `data_analyse.m`.

In [ ]:
# Reads the file this notebook just wrote in Part 1 -- no input needed. (Reading back
# from disk, rather than reusing the in-memory ResultMatrix, means this cell also works
# if you come back later and re-run just Part 2 against an existing Full_data.xlsx.)
ResultMatrix = pd.read_excel("Full_data.xlsx", header=None).to_numpy()


def get_row(matrix, wavelength_value):
    """Return columns 1: of the row whose column-0 value equals wavelength_value
    (mirrors MATLAB's find(ResultMatrix == value) trick for locating rows by their
    first-column value -- here, a wavelength or the -233333 voltage-row marker)."""
    idx = np.where(matrix[:, 0] == wavelength_value)[0]
    if len(idx) == 0:
        raise ValueError(f"Wavelength {wavelength_value} not found in first column of data.")
    return matrix[idx[0], 1:]


HighRow = get_row(ResultMatrix, isosbestic1)
LowRow = get_row(ResultMatrix, isosbestic2)
ApexRow = get_row(ResultMatrix, Apex)
Potential = get_row(ResultMatrix, -233333)   # the voltage row, found via its placeholder


In [ ]:
# Baseline-correct the apex absorbance using the two isosbestic points, then normalize
# to a 0-1 fraction spanning the first to the last voltage point.
Modified_Apex = ApexRow - (
    HighRow - (isosbestic1 - Apex) / (isosbestic1 - isosbestic2) * (HighRow - LowRow)
)

Fraction = (Modified_Apex - Modified_Apex[-1]) / (Modified_Apex[0] - Modified_Apex[-1])


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(Potential, Fraction, "o-")
plt.xlabel("Potential (mV)")
plt.ylabel("Fraction")
plt.title("Fraction vs Potential")
plt.show()


In [ ]:
Results = pd.DataFrame(np.vstack([Potential, Fraction]))
Results.to_excel("PotentialFraction.xlsx", header=False, index=False)
print("Saved PotentialFraction.xlsx")


## Part 3 — Fit Fraction vs. Potential with `lmfit`

Fits the one- or two-term Boltzmann/Nernst equation used in the original `data_analyse.m`:

`Fraction(V) = A1/(exp((V-Ema)*n1*39.585/1000)+1) + A2/(exp((V-Emb)*n2*39.585/1000)+1)`

This is the Python equivalent of sending your Potential/Fraction data to MATLAB's Curve Fitting
app. `lmfit` gives you a fit report (parameter values, standard errors, R², reduced
chi-square) very similar to what `cftool` shows.

`num_potentials` and the initial guesses (`Ema_guess`, `Emb_guess`, etc.) are all set in the
parameters cell at the top -- nothing to change here.

*(Requires `lmfit`: `pip install lmfit` if it isn't already installed.)*

In [ ]:
# Reads the file this notebook just wrote in Part 2 -- no input needed.
pf = pd.read_excel("PotentialFraction.xlsx", header=None).to_numpy()
x_data = pf[0, :]
y_data = pf[1, :]

CONST = 39.585 / 1000   # n*F/(R*T) at room temperature, in 1/mV -- the Nernst/Boltzmann constant


def one_potential(x, A1, Ema, n1):
    """Single-wave Boltzmann/Nernst fraction curve."""
    return A1 / (np.exp((x - Ema) * n1 * CONST) + 1)


def two_potential(x, A1, Ema, n1, A2, Emb, n2):
    """Sum of two overlapping Boltzmann/Nernst waves."""
    return (A1 / (np.exp((x - Ema) * n1 * CONST) + 1)
            + A2 / (np.exp((x - Emb) * n2 * CONST) + 1))


# Build the model and initial parameter guesses for whichever case num_potentials selects.
if num_potentials == 1:
    model = Model(one_potential)
    params = model.make_params(A1=A1_guess, Ema=Ema_guess, n1=n1_guess)
else:
    model = Model(two_potential)
    params = model.make_params(A1=A1_guess, Ema=Ema_guess, n1=n1_guess,
                                A2=A2_guess, Emb=Emb_guess, n2=n2_guess)

result = model.fit(y_data, params, x=x_data)
print(result.fit_report())


### 95% confidence intervals (matches MATLAB's `cftool` convention)

`lmfit`'s `stderr` above is a 1σ standard error, **not** the 95% CI that `cftool` reports by
default. `conf_interval()` computes a rigorous, profiled 95% CI for each parameter (better than
just scaling stderr by 1.96, especially if parameters are correlated -- common with two
overlapping potentials). Results are printed with a proper "±".

The final output file is named from `sweep_label` (`oxi.xlsx` or `red.xlsx`), and -- beyond the
95% CI table -- also includes the fitted equation and the Potential/Fraction data used for the
fit, so this one file is everything `plot_together.py` needs later to plot the oxidative and
reductive curves together.

In [ ]:
# Profile the likelihood around each parameter's best-fit value to get a proper,
# possibly-asymmetric 95% CI (sigmas=0.95 is interpreted as a probability, not a
# sigma count, since it's <= 1 -- see lmfit's conf_interval docs).
ci_results = conf_interval(result, result, sigmas=[0.95])

rows = []            # pretty, human-readable report (parameter, value ± error, CI range)
numeric_rows = []    # plain numeric values, for plot_together.py to reconstruct the curve
for name, param in result.params.items():
    values = sorted(v for _, v in ci_results[name])
    lower, upper = values[0], values[-1]
    ci95_error = (upper - lower) / 2   # symmetric half-width of the 95% CI, for the ± display
    rows.append({
        "parameter": name,
        "value ± 95% CI error": f"{param.value:.4g} ± {ci95_error:.4g}",
        "95% CI": f"(95% CI: [{lower:.4g}, {upper:.4g}])",
    })
    numeric_rows.append({"parameter": name, "value": param.value})

ci_df = pd.DataFrame(rows)
params_numeric_df = pd.DataFrame(numeric_rows)

for _, r in ci_df.iterrows():
    print(f"{r['parameter']} = {r['value ± 95% CI error']}   {r['95% CI']}")

# Human-readable fitted equation, with the fitted values plugged in -- saved alongside
# the parameters so oxi.xlsx/red.xlsx are self-documenting even without this notebook.
if num_potentials == 1:
    equation_str = (
        f"Fraction(V) = {result.params['A1'].value:.4g} / "
        f"(exp((V - {result.params['Ema'].value:.4g}) * {result.params['n1'].value:.4g} * 39.585/1000) + 1)"
    )
else:
    equation_str = (
        f"Fraction(V) = {result.params['A1'].value:.4g} / "
        f"(exp((V - {result.params['Ema'].value:.4g}) * {result.params['n1'].value:.4g} * 39.585/1000) + 1)"
        f" + {result.params['A2'].value:.4g} / "
        f"(exp((V - {result.params['Emb'].value:.4g}) * {result.params['n2'].value:.4g} * 39.585/1000) + 1)"
    )
print(f"\nFitted equation: {equation_str}")

equation_df = pd.DataFrame({"equation": [equation_str]})
fit_data_df = pd.DataFrame({"Potential": x_data, "Fraction": y_data})

# Write everything to one multi-sheet workbook named after sweep_label. Overwrites the
# file even if it already exists; the only way this can fail is if it's currently open
# elsewhere (e.g. in Excel), which locks it -- in that case fall back to a differently
# named file rather than losing the run's results.
output_path = f"{sweep_label}.xlsx"
try:
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        equation_df.to_excel(writer, sheet_name="equation", index=False)
        ci_df.to_excel(writer, sheet_name="parameters", index=False)
        params_numeric_df.to_excel(writer, sheet_name="parameters_numeric", index=False)
        fit_data_df.to_excel(writer, sheet_name="fit_data", index=False)
    print(f"\nSaved {output_path}")
except PermissionError:
    fallback = f"{sweep_label}_new.xlsx"
    with pd.ExcelWriter(fallback, engine="openpyxl") as writer:
        equation_df.to_excel(writer, sheet_name="equation", index=False)
        ci_df.to_excel(writer, sheet_name="parameters", index=False)
        params_numeric_df.to_excel(writer, sheet_name="parameters_numeric", index=False)
        fit_data_df.to_excel(writer, sheet_name="fit_data", index=False)
    print(f"\n'{output_path}' is open elsewhere (e.g. in Excel) and could not be "
          f"overwritten -- saved to '{fallback}' instead. Close the original file and "
          f"re-run this cell to overwrite it directly.")


In [ ]:
# Sanity-check plot: fitted curve over the actual data.
x_fit = np.linspace(x_data.min(), x_data.max(), 400)
y_fit = result.eval(x=x_fit)

plt.figure(figsize=(6, 4))
plt.plot(x_data, y_data, "o", label="Data")
plt.plot(x_fit, y_fit, "-", label="Fit")
plt.xlabel("Potential (mV)")
plt.ylabel("Fraction")
plt.title("Fraction vs Potential -- lmfit result")
plt.legend()
plt.show()
